# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Lecture 14: reducing dimension and finding groups

This notebook follows required Sections 11.1--11.5. Random projection
preserves distances probabilistically; SVD and PCA find deterministic
low-rank subspaces; $K$-means gives a discrete nearest-centre
representation. Their objectives are related through Euclidean
geometry but are not interchangeable.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits, make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(2026)
np.set_printoptions(precision=4, suppress=True)


## A quick random projection

Draw one $k\times d$ sign matrix $S$ and use the common map
$F(x)=Sx/\sqrt k$. For fixed $x$,
$\mathbb E|F(x)|^2=|x|^2$. Johnson--Lindenstrauss applies this one
common map to a fixed finite data set. Drawing a separate map for each
observation would not preserve pairwise differences.


In [ ]:
def random_sign_projection(X, k, rng):
    X = np.asarray(X, float)
    signs = rng.choice(np.array([-1.0, 1.0]), size=(k, X.shape[1]))
    return X @ signs.T / np.sqrt(k), signs

def pairwise_distances(X):
    differences = X[:, None, :] - X[None, :, :]
    distances = np.linalg.norm(differences, axis=2)
    return distances[np.triu_indices(len(X), k=1)]

X_high = rng.normal(size=(100, 200))
original = pairwise_distances(X_high)
plot_errors = None
for k in (10, 30, 80):
    projected, signs = random_sign_projection(X_high, k, rng)
    relative_error = np.abs(pairwise_distances(projected) / original - 1)
    print(
        f"k={k:2d}: median={np.median(relative_error):.3f}, "
        f"95%={np.quantile(relative_error, 0.95):.3f}, "
        f"maximum={relative_error.max():.3f}"
    )
    if k == 30:
        plot_errors = relative_error
plt.hist(plot_errors, bins=40)
plt.xlabel("relative pairwise-distance error for k=30")
plt.show()


The theorem's sufficient target dimension is conservative; this finite
diagnostic can work well below it. The simulation is not a replacement
for the simultaneous probability statement.

## Reconstructing data with centred SVD

NumPy returns the reduced decomposition
$A=U\operatorname{diag}(s)V^T$. PCA uses the centred matrix
$A=X-\bar x$. Its rank-$k$ reconstruction is
$A_k=(U_{:,1:k}s_{1:k})V_{1:k,:}^T$, and
$\|A-A_k\|_F^2=\sum_{j>k}s_j^2$. Add $\bar x$ to return to the
original coordinate scale.


In [ ]:
covariance = np.array([
    [2.0, 1.2, 0.4],
    [1.2, 1.5, 0.2],
    [0.4, 0.2, 0.3],
])
X_svd = rng.multivariate_normal([4.0, -2.0, 1.0], covariance, size=500)
sample_mean = X_svd.mean(axis=0)
A = X_svd - sample_mean
U, singular_values, Vt = np.linalg.svd(A, full_matrices=False)
full_reconstruction = (U * singular_values) @ Vt
k = 2
A_k = (U[:, :k] * singular_values[:k]) @ Vt[:k]
X_k = A_k + sample_mean
squared_error = np.linalg.norm(A - A_k, ord="fro") ** 2
tail_sum = np.sum(singular_values[k:] ** 2)
print("rank:", np.linalg.matrix_rank(A))
print("full reconstruction error:", np.linalg.norm(A - full_reconstruction))
print("rank-2 squared error/tail:", squared_error, tail_sum)
print("reconstruction shape:", X_k.shape)


Singular values are not empirical standard deviations by themselves.
With covariance $A^TA/n$, the standard deviation in principal
direction $j$ is $s_j/\sqrt n$.

## Checking the power method

For $B=A^TA$, $q_{t+1}=Bq_t/|Bq_t|$ approaches a leading
eigenvector if the largest eigenvalue is strictly larger than the
second and $q_0$ has a nonzero leading component. Sign is immaterial.
A small gap slows convergence; exact initial orthogonality prevents it.


In [ ]:
def power_errors(B, q0, leading_vector, iterations=60):
    q = np.asarray(q0, float)
    q /= np.linalg.norm(q)
    errors = []
    for _ in range(iterations):
        q = B @ q
        q /= np.linalg.norm(q)
        errors.append(1 - abs(q @ leading_vector))
    return np.array(errors)

Q, _ = np.linalg.qr(rng.normal(size=(3, 3)))
leading = Q[:, 0]
q0 = np.array([1.0, -0.4, 0.7])
for ratio in (0.50, 0.95):
    B = Q @ np.diag([1.0, ratio, 0.2]) @ Q.T
    errors = power_errors(B, q0, leading)
    plt.semilogy(np.maximum(errors, 1e-16),
                 label=fr"$\lambda_2/\lambda_1={ratio}$")
plt.xlabel("iteration")
plt.ylabel(r"$1-|q_t\cdot v_1|$")
plt.legend()
plt.show()


## PCA on the digits data

Scikit-learn PCA centres with the training mean. We fit only on
training images, transform held-out images with the same fit, and
reconstruct in the original 64-pixel space. Explained variance is a
property of the centred training matrix.


In [ ]:
digits = load_digits()
X_digits, y_digits = digits.data, digits.target
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_digits, y_digits, test_size=0.25, stratify=y_digits,
    random_state=2026
)
pca_digits = PCA(n_components=16, svd_solver="full")
train_scores = pca_digits.fit_transform(Xd_train)
test_scores = pca_digits.transform(Xd_test)
reconstruction = pca_digits.inverse_transform(test_scores)
print("scores/reconstruction:", train_scores.shape, reconstruction.shape)
print("variance fraction:", pca_digits.explained_variance_ratio_.sum())
print("held-out reconstruction MSE:",
      np.mean((Xd_test - reconstruction) ** 2))

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for j in range(8):
    axes[0, j].imshow(Xd_test[j].reshape(8, 8), cmap="gray")
    axes[1, j].imshow(reconstruction[j].reshape(8, 8), cmap="gray")
    axes[0, j].axis("off")
    axes[1, j].axis("off")
axes[0, 0].set_ylabel("original")
axes[1, 0].set_ylabel("PCA")
plt.tight_layout()
plt.show()


## What $K$-means minimises

For centres $\mu_1,\ldots,\mu_K$, the training objective is
$$
\Phi=\sum_i\min_j|x_i-\mu_j|^2.
$$
Lloyd assignment and centre updates never increase it, but need not
find the global minimum. Multiple restarts matter, and cluster numbers
are arbitrary. Because squared Euclidean distance depends on units,
scaling is a modelling decision and must be fitted without held-out
data.


In [ ]:
X_base, latent_group = make_blobs(
    n_samples=900,
    centers=[(-3, 0), (0, 0), (3, 0)],
    cluster_std=0.65,
    random_state=2026,
)
X_units = X_base.copy()
X_units[:, 1] *= 12
X_build, X_holdout, group_build, group_holdout = train_test_split(
    X_units, latent_group, test_size=0.20, random_state=2026
)
X_train, X_valid, group_train, group_valid = train_test_split(
    X_build, group_build, test_size=0.25, random_state=2026
)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

raw_fit = KMeans(n_clusters=3, n_init=20, random_state=2026).fit(X_train)
scaled_fit = KMeans(n_clusters=3, n_init=20,
                    random_state=2026).fit(X_train_scaled)
print("illustrative validation ARI, raw:",
      adjusted_rand_score(group_valid, raw_fit.predict(X_valid)))
print("illustrative validation ARI, scaled:",
      adjusted_rand_score(group_valid,
          scaled_fit.predict(X_valid_scaled)))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(X_valid[:, 0], X_valid[:, 1],
                c=raw_fit.predict(X_valid), s=12)
axes[0].set_title("raw units")
axes[1].scatter(X_valid[:, 0], X_valid[:, 1],
                c=scaled_fit.predict(X_valid_scaled), s=12)
axes[1].set_title("training-only scaling")
plt.tight_layout()
plt.show()


Latent labels exist only because this is a simulation; they are used
for illustration, never fitting. In real clustering there are no
response labels.

Training distortion cannot select $K$: it cannot increase as centres
are added, and $K=n$ can make it zero. Domain constraints, stability,
or a downstream purpose are needed. A validation curve can reveal the
trade-off, but minimizing distortion alone favours larger $K$.


In [ ]:
single_start = []
for seed in range(12):
    fit = KMeans(n_clusters=3, init="random", n_init=1,
                 random_state=seed).fit(X_train_scaled)
    single_start.append(fit.inertia_)
many_starts = KMeans(n_clusters=3, n_init=20,
                     random_state=2026).fit(X_train_scaled)
print("single-start inertia range:", min(single_start), max(single_start))
print("20-start retained inertia:", many_starts.inertia_)

def mean_distortion(model, X):
    return np.mean(np.min(model.transform(X) ** 2, axis=1))

k_values = np.arange(1, 7)
training_curve, validation_curve = [], []
for K in k_values:
    fit = KMeans(n_clusters=K, n_init=20,
                 random_state=2026).fit(X_train_scaled)
    training_curve.append(fit.inertia_ / len(X_train_scaled))
    validation_curve.append(mean_distortion(fit, X_valid_scaled))
plt.plot(k_values, training_curve, "o-", label="training")
plt.plot(k_values, validation_curve, "s--", label="validation")
plt.xlabel("K")
plt.ylabel("mean squared nearest-centre distance")
plt.legend()
plt.show()


For this designed example we fix $K=3$ before final assessment.
Scaling and clustering are refitted on all development data, then
distortion is computed once on the untouched holdout. This estimates
$\mathbb E[\min_j|X-\mu_j|^2]$ for the fitted centres.

The Gaussian generator is unbounded, so the bounded-support Hoeffding
result in the notes does not apply. Standardizing observations does not
impose an almost-sure bound on future data.


In [ ]:
final_scaler = StandardScaler().fit(X_build)
X_build_scaled = final_scaler.transform(X_build)
X_holdout_scaled = final_scaler.transform(X_holdout)
final_kmeans = KMeans(n_clusters=3, n_init=30,
                      random_state=2026).fit(X_build_scaled)
print("untouched held-out distortion:",
      mean_distortion(final_kmeans, X_holdout_scaled))


### How mixture models help us think about $K$-means

If a hidden label $Z=j$ has conditional mean $m_j$, the centroid
identity makes $m_j$ optimal for squared error when hidden labels are
known. Ordinary $K$-means instead assigns by nearest fitted centre.
Overlap, unequal weights or spreads, and nonspherical shapes can make
fitted clusters differ from mixture components. There is no general
recovery guarantee.

## Compare PCA and $K$-means

PCA gives continuous scores and minimizes subspace reconstruction
error. $K$-means gives discrete assignments and minimizes
nearest-centre distortion. The cell fits both clustering routes
without held-out digit labels; labels enter only in ARI evaluation.

Change the component count and restart count. Explain why PCA error and
cluster distortion are different; why distortions in original and PCA
spaces are not directly comparable; and whether PCA helps this example
without claiming general digit-population recovery.


In [ ]:
n_components, n_init = 20, 20
digit_scaler = StandardScaler().fit(Xd_train)
Xdt = digit_scaler.transform(Xd_train)
Xdv = digit_scaler.transform(Xd_test)

direct = KMeans(n_clusters=10, n_init=n_init,
                random_state=2026).fit(Xdt)
direct_ari = adjusted_rand_score(yd_test, direct.predict(Xdv))

digit_pca = PCA(n_components=n_components, random_state=2026).fit(Xdt)
Xdt_pca, Xdv_pca = digit_pca.transform(Xdt), digit_pca.transform(Xdv)
after_pca = KMeans(n_clusters=10, n_init=n_init,
                   random_state=2026).fit(Xdt_pca)
pca_ari = adjusted_rand_score(yd_test, after_pca.predict(Xdv_pca))
print("held-out ARI, scaled pixels:", direct_ari)
print("held-out ARI, PCA scores:", pca_ari)
print("PCA variance fraction:", digit_pca.explained_variance_ratio_.sum())
